In [2]:
import numpy as np
from scipy.stats import norm
import config

def bs_call(S, K, T, r, sigma):
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)

def bs_put(S, K, T, r, sigma):
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return K * np.exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1)

def chooser_option_price(S, K, T1, T2, r, sigma):
    call_price = bs_call(S, K, T2, r, sigma)
    K_put = K * np.exp(-r * (T2 - T1))
    put_price = bs_put(S, K_put, T1, r, sigma)
    return call_price + put_price

# 使用配置文件中的参数
price = chooser_option_price(config.S, config.K, config.T1, config.T2, config.r, config.sigma)

print(f"Parameters Loaded: Strike=${config.K}, T1={config.T1}, T2={config.T2}")
print(f"BSM Chooser Option Price: ${price:.4f}")

Parameters Loaded: Strike=$150.0, T1=0.5, T2=1.0
BSM Chooser Option Price: $20.7770


### Initial Comparisons & Validation Note

**Model Result:** The BSM Chooser Option price is calculated as **$20.7770** for a stock at $150, strike at $150, T1=0.5, T2=1.0, r=5%, and volatility of 20%.

**Sanity Check:** As theoretically expected, this Chooser Option price ($20.77) is strictly greater than a standard 1-year European Call (~$12) and a standard 1-year European Put (~$8). This confirms that the model correctly prices the "flexibility" premium of being able to choose between Call and Put.

**Comparison to Paper:** The original paper's reported results will be directly compared in **Week 4** once we integrate actual CME transaction data. The current step successfully establishes the mathematical foundation for the baseline model.

In [4]:
# ====== Greeks 分析 (Delta, Gamma, Vega) ======
# 使用数值微分法 (Finite Difference) 计算 Greeks
# 因为 Chooser Option 是组合期权，直接用微小扰动来求导最直观

eps = 1e-4 # 极小扰动值

# 1. Delta: 股票价格 S 变动 1 单位带来的价格变动
price_up_S = chooser_option_price(config.S + eps, config.K, config.T1, config.T2, config.r, config.sigma)
price_down_S = chooser_option_price(config.S - eps, config.K, config.T1, config.T2, config.r, config.sigma)
delta = (price_up_S - price_down_S) / (2 * eps)

# 2. Gamma: Delta 随股票价格变化的速率
gamma = (price_up_S - 2 * price + price_down_S) / (eps**2)

# 3. Vega: 波动率 sigma 变动 1% (0.01) 带来的价格变动
price_up_V = chooser_option_price(config.S, config.K, config.T1, config.T2, config.r, config.sigma + 0.01)
vega = price_up_V - price

print("===== Chooser Option Greeks Analysis =====")
print(f"Current Option Price: ${price:.4f}")
print(f"Delta (Δ): {delta:.4f}  (If stock goes up $1, option goes up ${delta:.4f})")
print(f"Gamma (Γ): {gamma:.4f}")
print(f"Vega  (ν): {vega:.4f}   (If volatility goes up 1%, option goes up ${vega:.4f})")

===== Chooser Option Greeks Analysis =====
Current Option Price: $20.7770
Delta (Δ): 0.3011  (If stock goes up $1, option goes up $0.3011)
Gamma (Γ): 0.0297
Vega  (ν): 0.9514   (If volatility goes up 1%, option goes up $0.9514)


### Greeks Analysis Results

The Greeks for the Chooser Option were calculated using the finite difference method:
- **Delta (Δ) = 0.3011**: If the underlying stock price increases by $1, the chooser option price increases by approximately $0.3011.
- **Gamma (Γ) = 0.0297**: The rate of change of Delta with respect to the stock price.
- **Vega (ν) = 0.9514**: If implied volatility increases by 1%, the chooser option price increases by approximately $0.95.

These values confirm that the chooser option has a blended sensitivity profile (as it combines both call and put features) and is highly sensitive to volatility changes, which is expected for long-dated options.